# Tokenizer

SentencePiece BPE tokenizer used by the pretraining pipeline. Run this notebook directly in Colab to train, encode, and decode text interactively.

In [ ]:
%pip install -q sentencepiece numpy

In [ ]:
from __future__ import annotations

from pathlib import Path

import sentencepiece as spm

In [ ]:
class Tokenizer:
    """
    Wraps a SentencePiece model for fast in-memory encode/decode.

    Load with Tokenizer.from_pretrained(path) or create fresh with train().
    """

    def __init__(self, model_path: str | Path):
        self.model_path = Path(model_path)
        self._sp: spm.SentencePieceProcessor | None = None

    def _ensure_loaded(self) -> spm.SentencePieceProcessor:
        if self._sp is None:
            self._sp = spm.SentencePieceProcessor()
            self._sp.load(str(self.model_path))
        return self._sp

    @property
    def vocab_size(self) -> int:
        return self._ensure_loaded().get_piece_size()

    def encode(self, text: str) -> list[int]:
        """Encode string to list of token IDs."""
        return self._ensure_loaded().encode(text, out_type=int)

    def decode(self, tokens: list[int]) -> str:
        """Decode list of token IDs to string."""
        return self._ensure_loaded().decode(tokens)

    @classmethod
    def from_pretrained(cls, model_path: str | Path) -> "Tokenizer":
        """Load an existing tokenizer from a .model file."""
        return cls(model_path)

    @classmethod
    def train(
        cls,
        text_path: str | Path,
        vocab_size: int = 500,
        *,
        output_dir: str | Path = ".",
        name: str = "tokenizer",
    ) -> "Tokenizer":
        """Train a SentencePiece BPE model on text_path and return a Tokenizer."""
        text_path = Path(text_path)
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)
        model_prefix = str(output_dir / name)

        spm.SentencePieceTrainer.train(
            input=str(text_path),
            model_prefix=model_prefix,
            vocab_size=vocab_size,
            model_type="bpe",
            pad_id=0,
            unk_id=1,
            bos_id=-1,
            eos_id=-1,
            pad_piece="<pad>",
            unk_piece="<unk>",
            byte_fallback=True,
            add_dummy_prefix=False,
        )

        model_path = f"{model_prefix}.model"
        print(f"[tokenizer] trained - saved model -> {model_path}")
        return cls(model_path)

## Quick Test

Set `CORPUS_PATH` to a text file in the repo or an uploaded Colab file, then train a small tokenizer and round-trip a sample.

In [ ]:
CORPUS_PATH = Path("../data/van-life-story.txt")
if not CORPUS_PATH.exists():
    CORPUS_PATH = Path("data/van-life-story.txt")

tok = Tokenizer.train(CORPUS_PATH, vocab_size=1024, output_dir="data", name="sp")
ids = tok.encode("Hello, world!")
print(ids)
print(tok.decode(ids))